# QuickPay FinTech — Data Pipeline
**Assignment:** QuickPay FinTech Operations Case Study  
**Notebook covers:**
- Part 3 — Python Reconciliation Workflow (Ledger vs Gateway)
- Part 4 — JSON Normalization (API Response)
- Part 5 Prep — Dashboard Output CSVs

> ⚙️ **Google Colab users:** Upload all raw CSV/JSON files to the Colab session storage before running, or mount Google Drive and update file paths accordingly.


## Cell 1 — Environment Setup & Library Imports

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────
# Importing all required libraries upfront.
# pandas  : data loading, manipulation, merging
# numpy   : numeric checks and tolerances
# json    : reading and writing JSON files
# datetime: timestamp formatting

import pandas as pd
import numpy as np
import json
from datetime import datetime

# Suppress non-critical warnings for clean output
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


: 

## Cell 2 — File Path Configuration

In [ ]:
# ── Cell 2: File Path Configuration ───────────────────────────────────
# Centralise all file paths here so you only need to update one place
# if you move files (e.g. mounting Google Drive on Colab).
#
# Google Colab users: replace the paths below with your Drive path, e.g.
#   BASE_RAW = '/content/drive/MyDrive/quickpay/01_data/raw/'

BASE_RAW       = '/Users/punith/Downloads/'          # folder containing raw CSV/JSON files
BASE_PROCESSED = '/Users/punith/Downloads/Output'          # folder where processed outputs will be saved

# Input files
PATH_LEDGER      = BASE_RAW + 'ledger.csv'
PATH_GATEWAY     = BASE_RAW + 'gateway.csv'
PATH_CLEANED_TX  = BASE_RAW + 'cleaned_transactions.csv'
PATH_API_JSON    = BASE_RAW + 'api_response_sample.json'

# Output files — Part 3
PATH_MISSING_IN_GATEWAY   = BASE_PROCESSED + 'missing_in_gateway.csv'
PATH_MISSING_IN_LEDGER    = BASE_PROCESSED + 'missing_in_ledger.csv'
PATH_AMOUNT_MISMATCHES    = BASE_PROCESSED + 'amount_mismatches.csv'
PATH_STATUS_MISMATCHES    = BASE_PROCESSED + 'status_mismatches.csv'
PATH_RECON_REPORT         = BASE_PROCESSED + 'reconciliation_report.csv'
PATH_SUMMARY_METRICS      = 'summary_metrics.json'

# Output files — Part 4
PATH_API_NORMALIZED = BASE_PROCESSED + 'api_normalized.csv'

# Output files — Part 5 dashboard prep
PATH_DAILY_SUMMARY    = BASE_PROCESSED + 'daily_summary.csv'
PATH_PAYMENT_BREAKDOWN = BASE_PROCESSED + 'payment_method_breakdown.csv'
PATH_REGION_BREAKDOWN  = BASE_PROCESSED + 'region_breakdown.csv'
PATH_MERCHANT_PERF     = BASE_PROCESSED + 'merchant_performance_summary.csv'

print("✅ File paths configured.")


---
## Part 3 — Reconciliation Workflow
### Cell 3 — Load Ledger & Gateway Files

In [ ]:
# ── Cell 3: Load Ledger and Gateway Files ─────────────────────────────
# ledger  = QuickPay's internal record of what SHOULD have happened
# gateway = External bank/payment processor record of what DID happen
# Reconciliation = finding every place these two disagree

ledger  = pd.read_csv(PATH_LEDGER)
gateway = pd.read_csv(PATH_GATEWAY)

print("=" * 55)
print("LEDGER  — Internal Records")
print("=" * 55)
print(f"Shape   : {ledger.shape[0]} rows × {ledger.shape[1]} columns")
print(f"Columns : {list(ledger.columns)}")
print()
print(ledger.to_string(index=False))

print()
print("=" * 55)
print("GATEWAY — External Bank Records")
print("=" * 55)
print(f"Shape   : {gateway.shape[0]} rows × {gateway.shape[1]} columns")
print(f"Columns : {list(gateway.columns)}")
print()
print(gateway.to_string(index=False))


### Cell 4 — Data Quality Checks (Nulls & Duplicates)

In [ ]:
# ── Cell 4: Data Quality Checks ───────────────────────────────────────
# Before reconciling, we validate both files.
# Nulls in key columns or duplicate transaction_ids would cause
# false matches and corrupt the reconciliation output.

def quality_report(df, name):
    print(f"{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  Rows            : {len(df)}")
    print(f"  Columns         : {df.shape[1]}")
    print()

    # Null check per column
    nulls = df.isnull().sum()
    total_nulls = nulls.sum()
    print(f"  NULL values per column:")
    for col, cnt in nulls.items():
        flag = ' ⚠️' if cnt > 0 else ' ✅'
        print(f"    {col:<20} : {cnt}{flag}")
    print(f"  Total nulls     : {total_nulls} {'⚠️  — needs attention' if total_nulls > 0 else '✅'}")
    print()

    # Duplicate check
    dup_count = df['transaction_id'].duplicated().sum()
    print(f"  Duplicate transaction_ids : {dup_count} {'⚠️  — will cause bad joins' if dup_count > 0 else '✅'}")
    print()

    # Data type summary
    print(f"  Column dtypes:")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<20} : {dtype}")
    print()

quality_report(ledger,  "LEDGER")
quality_report(gateway, "GATEWAY")

print("=" * 50)
print("Quality check complete — both files are clean.")
print("No nulls. No duplicate transaction IDs.")
print("Safe to proceed with reconciliation.")


### Cell 5 — Records Missing in Gateway

In [ ]:
# ── Cell 5: Records Missing in Gateway ────────────────────────────────
# These are transactions recorded internally in the ledger
# but NOT acknowledged by the payment gateway.
#
# Risk: QuickPay's books show these as complete, but the bank
# never processed them. The merchant may have been paid from
# internal funds while the gateway settlement never happened.

ledger_ids  = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

# Set difference: IDs that exist in ledger but not in gateway
missing_in_gateway_ids = ledger_ids - gateway_ids

print(f"Transaction IDs in ledger    : {len(ledger_ids)}")
print(f"Transaction IDs in gateway   : {len(gateway_ids)}")
print(f"Missing in gateway           : {len(missing_in_gateway_ids)}")
print(f"Missing IDs                  : {sorted(missing_in_gateway_ids)}")
print()

# Pull the full ledger rows for these IDs
missing_in_gateway = ledger[
    ledger['transaction_id'].isin(missing_in_gateway_ids)
].copy().reset_index(drop=True)

# Tag with reconciliation issue type for the final report
missing_in_gateway['reconciliation_issue'] = 'missing_in_gateway'
missing_in_gateway['amount_gateway']       = None
missing_in_gateway['status_gateway']       = None

print("Rows missing in gateway:")
print(missing_in_gateway.to_string(index=False))

# Save output
missing_in_gateway.to_csv(PATH_MISSING_IN_GATEWAY, index=False)
print(f"\n✅ Saved: missing_in_gateway.csv  ({len(missing_in_gateway)} rows)")


### Cell 6 — Records Missing in Ledger

In [ ]:
# ── Cell 6: Records Missing in Ledger ─────────────────────────────────
# These are transactions the gateway processed that have no matching
# internal record in the ledger.
#
# Risk: Unbooked revenue. The bank settled a payment that QuickPay
# never recorded. Could also indicate ghost transactions, where
# money moved without internal authorisation.

# Set difference: IDs in gateway but not in ledger
missing_in_ledger_ids = gateway_ids - ledger_ids

print(f"Missing in ledger            : {len(missing_in_ledger_ids)}")
print(f"Missing IDs                  : {sorted(missing_in_ledger_ids)}")
print()

# Pull the full gateway rows for these IDs
missing_in_ledger = gateway[
    gateway['transaction_id'].isin(missing_in_ledger_ids)
].copy().reset_index(drop=True)

# Tag with reconciliation issue type
missing_in_ledger['reconciliation_issue'] = 'missing_in_ledger'
missing_in_ledger['amount_ledger']        = None
missing_in_ledger['status_ledger']        = None

print("Rows missing in ledger:")
print(missing_in_ledger.to_string(index=False))

# Save output
missing_in_ledger.to_csv(PATH_MISSING_IN_LEDGER, index=False)
print(f"\n✅ Saved: missing_in_ledger.csv  ({len(missing_in_ledger)} rows)")


### Cell 7 — Merge Common Records for Side-by-Side Comparison

In [ ]:
# ── Cell 7: Merge Common Transaction IDs ──────────────────────────────
# Inner join on transaction_id keeps only rows that exist in BOTH files.
# Suffixes _ledger and _gateway let us compare each field side by side.
# This merged table is the foundation for all mismatch checks.

common_ids = ledger_ids & gateway_ids
print(f"Transaction IDs common to both files : {len(common_ids)}")
print(f"IDs                                  : {sorted(common_ids)}")
print()

# Inner merge on transaction_id
merged = pd.merge(
    ledger[ledger['transaction_id'].isin(common_ids)],
    gateway[gateway['transaction_id'].isin(common_ids)],
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
).reset_index(drop=True)

print(f"Merged shape: {merged.shape[0]} rows × {merged.shape[1]} columns")
print()
print("Side-by-side comparison (ledger vs gateway):")
display_cols = [
    'transaction_id',
    'amount_usd_ledger', 'amount_usd_gateway',
    'status_ledger',     'status_gateway',
    'merchant_id_ledger','payment_method_ledger'
]
print(merged[display_cols].to_string(index=False))


### Cell 8 — Amount Mismatches

In [ ]:
# ── Cell 8: Amount Mismatches ─────────────────────────────────────────
# Flag rows where the USD amount in ledger differs from the gateway.
# We use a tolerance of $0.01 to handle floating-point representation
# differences. Any gap larger than 1 cent is a real discrepancy.
#
# Found: R002 — ledger $850 vs gateway $900  (gateway $50 higher)
#        R008 — ledger $640 vs gateway $600  (ledger $40 higher)

# Compute absolute difference between the two amount columns
merged['amount_diff'] = (
    merged['amount_usd_ledger'] - merged['amount_usd_gateway']
).round(2)

merged['amount_diff_abs'] = merged['amount_diff'].abs()

# Flag as mismatch if difference exceeds 1 cent
amount_mismatches = merged[
    merged['amount_diff_abs'] > 0.01
].copy().reset_index(drop=True)

amount_mismatches['reconciliation_issue'] = 'amount_mismatch'

print(f"Amount mismatches found : {len(amount_mismatches)}")
print()

if len(amount_mismatches) > 0:
    display = amount_mismatches[[
        'transaction_id',
        'transaction_date_ledger',
        'merchant_id_ledger',
        'amount_usd_ledger',
        'amount_usd_gateway',
        'amount_diff',
        'reconciliation_issue'
    ]].rename(columns={
        'transaction_date_ledger' : 'transaction_date',
        'merchant_id_ledger'      : 'merchant_id'
    })
    print(display.to_string(index=False))
    print()
    print(f"  R002 — Ledger shows $850.00, gateway shows $900.00 → $50.00 gap (gateway higher)")
    print(f"  R008 — Ledger shows $640.00, gateway shows $600.00 → $40.00 gap (ledger higher)")
    print(f"  Total value at risk from amount discrepancies: ${amount_mismatches['amount_diff_abs'].sum():.2f}")

# Save output
amount_mismatches.to_csv(PATH_AMOUNT_MISMATCHES, index=False)
print(f"\n✅ Saved: amount_mismatches.csv  ({len(amount_mismatches)} rows)")


### Cell 9 — Status Mismatches

In [ ]:
# ── Cell 9: Status Mismatches ─────────────────────────────────────────
# Flag rows where the transaction status in ledger differs from gateway.
# A status mismatch is often more serious than an amount mismatch —
# 'success' in ledger vs 'failed' in gateway means we believe a payment
# settled when the bank says it did not.
#
# Found: R005 — ledger 'success' vs gateway 'failed'
#        (QuickPay thinks it settled; the bank says it failed)

# Compare status columns
status_mismatches = merged[
    merged['status_ledger'] != merged['status_gateway']
].copy().reset_index(drop=True)

status_mismatches['reconciliation_issue'] = 'status_mismatch'

print(f"Status mismatches found : {len(status_mismatches)}")
print()

if len(status_mismatches) > 0:
    display = status_mismatches[[
        'transaction_id',
        'transaction_date_ledger',
        'merchant_id_ledger',
        'amount_usd_ledger',
        'status_ledger',
        'status_gateway',
        'reconciliation_issue'
    ]].rename(columns={
        'transaction_date_ledger' : 'transaction_date',
        'merchant_id_ledger'      : 'merchant_id',
        'amount_usd_ledger'       : 'amount_usd'
    })
    print(display.to_string(index=False))
    print()
    print("  ⚠️  R005: Ledger records 'success' but gateway recorded 'failed'.")
    print("     This means QuickPay may have credited the merchant for a payment")
    print("     that never settled. Amount at risk: $7,200.00")
else:
    print("  ✅ No status mismatches found.")

# Save output
status_mismatches.to_csv(PATH_STATUS_MISMATCHES, index=False)
print(f"\n✅ Saved: status_mismatches.csv  ({len(status_mismatches)} rows)")


### Cell 10 — Final Reconciliation Report

In [ ]:
# ── Cell 10: Final Reconciliation Report ──────────────────────────────
# Combines all four issue types into a single unified report.
# Each row = one transaction with one issue label.
# A transaction can appear multiple times if it has multiple issues
# (e.g. both an amount mismatch AND a status mismatch).
# The ops team uses this report to prioritise follow-up actions.

# ── Build each section ──────────────────────────────────────────────
# Section A: Missing in gateway
section_a = missing_in_gateway[[
    'transaction_id','transaction_date','merchant_id',
    'amount_usd','status','payment_method','reconciliation_issue'
]].copy()

# Section B: Missing in ledger
section_b = missing_in_ledger[[
    'transaction_id','transaction_date','merchant_id',
    'amount_usd','status','payment_method','reconciliation_issue'
]].copy()

# Section C: Amount mismatches — use ledger as source of truth for metadata
section_c = pd.DataFrame({
    'transaction_id'      : amount_mismatches['transaction_id'],
    'transaction_date'    : amount_mismatches['transaction_date_ledger'],
    'merchant_id'         : amount_mismatches['merchant_id_ledger'],
    'amount_usd'          : amount_mismatches['amount_usd_ledger'],
    'status'              : amount_mismatches['status_ledger'],
    'payment_method'      : amount_mismatches['payment_method_ledger'],
    'reconciliation_issue': amount_mismatches['reconciliation_issue'],
    'amount_gateway'      : amount_mismatches['amount_usd_gateway'],
    'amount_diff'         : amount_mismatches['amount_diff']
})

# Section D: Status mismatches
section_d = pd.DataFrame({
    'transaction_id'      : status_mismatches['transaction_id'],
    'transaction_date'    : status_mismatches['transaction_date_ledger'],
    'merchant_id'         : status_mismatches['merchant_id_ledger'],
    'amount_usd'          : status_mismatches['amount_usd_ledger'],
    'status'              : status_mismatches['status_ledger'],
    'payment_method'      : status_mismatches['payment_method_ledger'],
    'reconciliation_issue': status_mismatches['reconciliation_issue'],
    'status_gateway'      : status_mismatches['status_gateway']
})

# ── Concatenate all sections ────────────────────────────────────────
recon_report = pd.concat(
    [section_a, section_b, section_c, section_d],
    ignore_index=True, sort=False
)

# Fill NaN for columns that not all sections populate
recon_report = recon_report.fillna('')

# Sort for readability
recon_report = recon_report.sort_values(
    ['reconciliation_issue','transaction_id']
).reset_index(drop=True)

print(f"Final Reconciliation Report — {len(recon_report)} total issue rows")
print()
print(recon_report.to_string(index=False))

# Issue summary table
print()
print("=" * 50)
print("Issue Summary:")
print(recon_report['reconciliation_issue'].value_counts().to_string())

# Save output
recon_report.to_csv(PATH_RECON_REPORT, index=False)
print(f"\n✅ Saved: reconciliation_report.csv  ({len(recon_report)} rows)")


### Cell 11 — Summary Metrics JSON

In [ ]:
# ── Cell 11: Summary Metrics JSON ─────────────────────────────────────
# Generates the required summary_metrics.json file.
# amount_at_risk = total USD value of all transactions involved in
# any reconciliation issue (using the higher of ledger vs gateway).

# Collect all transaction IDs that have any issue
all_issue_ids = (
    set(missing_in_gateway['transaction_id']) |
    set(missing_in_ledger['transaction_id'])  |
    set(amount_mismatches['transaction_id'])  |
    set(status_mismatches['transaction_id'])
)

# Amount at risk: sum from whichever source has the higher value
ledger_at_risk  = ledger[
    ledger['transaction_id'].isin(all_issue_ids)
]['amount_usd'].sum()

gateway_at_risk = gateway[
    gateway['transaction_id'].isin(all_issue_ids)
]['amount_usd'].sum()

summary_metrics = {
    "total_ledger_rows"          : len(ledger),
    "total_gateway_rows"         : len(gateway),
    "missing_in_gateway_count"   : len(missing_in_gateway),
    "missing_in_ledger_count"    : len(missing_in_ledger),
    "amount_mismatch_count"      : len(amount_mismatches),
    "status_mismatch_count"      : len(status_mismatches),
    "reconciliation_issue_count" : len(all_issue_ids),
    "ledger_total_amount"        : round(ledger['amount_usd'].sum(), 2),
    "gateway_total_amount"       : round(gateway['amount_usd'].sum(), 2),
    "amount_at_risk"             : round(max(ledger_at_risk, gateway_at_risk), 2)
}

print("Summary Metrics:")
print("-" * 42)
for k, v in summary_metrics.items():
    print(f"  {k:<35} : {v}")

# Save JSON
with open(PATH_SUMMARY_METRICS, 'w') as f:
    json.dump(summary_metrics, f, indent=2)

print(f"\n✅ Saved: summary_metrics.json")


---
## Part 4 — JSON Normalization

The `api_response_sample.json` file represents a real-world API response from the QuickPay Settlement API.  
It is **nested**: each batch contains a merchant object and an array of settlements, each of which contains a bank object.  
We flatten this into **one row per settlement** — a clean tabular format ready for analysis.


### Cell 12 — Load & Inspect JSON Structure

In [ ]:
# ── Cell 12: Load and Inspect JSON ───────────────────────────────────
# Understanding the nesting depth before flattening prevents
# silent data loss. We print the structure level by level.

with open(PATH_API_JSON, 'r') as f:
    api_data = json.load(f)

print("Top-level keys   :", list(api_data.keys()))
print("Source           :", api_data.get('source'))
print("Generated at     :", api_data.get('generated_at'))
print(f"Number of batches: {len(api_data['batches'])}")
print()

# Walk structure to show nesting
for batch in api_data['batches']:
    print(f"  Batch ID  : {batch['batch_id']}")
    print(f"  Merchant  : {batch['merchant']['merchant_id']} — {batch['merchant']['merchant_name']} ({batch['merchant']['region']})")
    print(f"  Settlements ({len(batch['settlements'])}):")
    for s in batch['settlements']:
        print(f"    {s['settlement_id']} | ${s['amount_usd']:>8.2f} | {s['status']:<8} | "
              f"{s['processed_at']} | Bank: {s['bank']['name']} ({s['bank']['country']})")
    print()

print("Nesting structure: batches → merchant (dict) → settlements (list) → bank (dict)")
print("Target           : one flat row per settlement = 6 total rows")


### Cell 13 — Flatten Nested JSON to Tabular Format

In [ ]:
# ── Cell 13: Flatten Nested JSON to Tabular Format ───────────────────
# Loop through batches → settlements, pulling all nested fields
# into a flat dictionary. Each dict becomes one DataFrame row.

rows = []

for batch in api_data['batches']:
    # Extract batch-level fields
    batch_id      = batch['batch_id']
    merchant_id   = batch['merchant']['merchant_id']
    merchant_name = batch['merchant']['merchant_name']
    region        = batch['merchant']['region']

    for settlement in batch['settlements']:
        row = {
            'batch_id'        : batch_id,
            'merchant_id'     : merchant_id,
            'merchant_name'   : merchant_name,
            'merchant_region' : region,
            'settlement_id'   : settlement['settlement_id'],
            'amount_usd'      : settlement['amount_usd'],
            'status'          : settlement['status'],
            'processed_at'    : settlement['processed_at'],
            'bank_name'       : settlement['bank']['name'],
            'bank_country'    : settlement['bank']['country'],
        }
        rows.append(row)

api_df = pd.DataFrame(rows)

print(f"Flattened shape : {api_df.shape[0]} rows × {api_df.shape[1]} columns")
print(f"Columns         : {list(api_df.columns)}")
print()
print(api_df.to_string(index=False))


### Cell 14 — Clean, Type-Cast & Save Normalized Output

In [ ]:
# ── Cell 14: Clean and Save Normalized API Data ───────────────────────
# Steps:
#   1. Convert processed_at from ISO 8601 with UTC timezone to
#      clean datetime string (YYYY-MM-DD HH:MM:SS)
#   2. Ensure amount_usd is float (already is, but explicit cast)
#   3. Lowercase status values for consistency with transactions table
#   4. Enforce final column order
#   5. Validate: check for nulls, check settlement_id uniqueness

# Step 1 — Convert timestamp
api_df['processed_at'] = pd.to_datetime(
    api_df['processed_at'], utc=True
).dt.strftime('%Y-%m-%d %H:%M:%S')

# Step 2 — Numeric cast
api_df['amount_usd'] = api_df['amount_usd'].astype(float)

# Step 3 — Lowercase status
api_df['status'] = api_df['status'].str.lower().str.strip()

# Step 4 — Enforce column order
api_df = api_df[[
    'batch_id', 'merchant_id', 'merchant_name', 'merchant_region',
    'settlement_id', 'amount_usd', 'status',
    'processed_at', 'bank_name', 'bank_country'
]]

# Step 5 — Validate
print("Validation checks:")
print(f"  Null values     : {api_df.isnull().sum().sum()} ✅")
print(f"  Unique settle.  : {api_df['settlement_id'].nunique()} / {len(api_df)} ✅")
print(f"  Status values   : {api_df['status'].unique()}")
print(f"  Date format ok  : {api_df['processed_at'].iloc[0]}")
print()

print("Final Normalized API Table:")
print(api_df.to_string(index=False))

# Save
api_df.to_csv(PATH_API_NORMALIZED, index=False)
print(f"\n✅ Saved: api_normalized.csv  ({len(api_df)} rows)")


---
## Part 5 Prep — Dashboard Output Files

The four CSVs below are generated from `cleaned_transactions.csv`  
and used as data sources in the Looker Studio dashboard.


### Cell 15 — Load Cleaned Transactions

In [ ]:
# ── Cell 15: Load cleaned_transactions for Dashboard Prep ─────────────
ct = pd.read_csv(PATH_CLEANED_TX)

print(f"Shape   : {ct.shape}")
print(f"Columns : {list(ct.columns)}")
print()
print(ct[['transaction_id','transaction_date','merchant_name',
          'gateway_region','amount_usd','status',
          'payment_method','risk_score']].head(5).to_string(index=False))


### Cell 16 — daily_summary.csv

In [ ]:
# ── Cell 16: Daily Summary ─────────────────────────────────────────────
# One row per date. Powers the trend/time-series chart in the dashboard.

daily = ct.groupby('transaction_date').agg(
    total_transactions      =('transaction_id', 'count'),
    total_gmv_usd           =('amount_usd',     'sum'),
    successful_transactions =('status', lambda x: (x == 'captured').sum()),
    failed_transactions     =('status', lambda x: (x == 'failed').sum()),
    chargeback_count        =('status', lambda x: (x == 'chargeback').sum()),
    avg_risk_score          =('risk_score', 'mean'),
    high_value_count        =('high_value_flag', 'sum'),
    high_risk_count         =('high_risk_flag',  'sum'),
).reset_index()

# Captured GMV (only settled revenue)
captured_gmv = ct[ct['status'] == 'captured'].groupby(
    'transaction_date')['amount_usd'].sum().reset_index()
captured_gmv.columns = ['transaction_date', 'captured_gmv_usd']
daily = daily.merge(captured_gmv, on='transaction_date', how='left').fillna(0)

# Derived metrics
daily['success_rate_pct'] = (
    daily['successful_transactions'] / daily['total_transactions'] * 100
).round(2)
daily['total_gmv_usd']    = daily['total_gmv_usd'].round(2)
daily['captured_gmv_usd'] = daily['captured_gmv_usd'].round(2)
daily['avg_risk_score']   = daily['avg_risk_score'].round(2)

# Final column order
daily = daily[[
    'transaction_date', 'total_transactions', 'total_gmv_usd',
    'captured_gmv_usd', 'successful_transactions', 'failed_transactions',
    'chargeback_count', 'success_rate_pct', 'avg_risk_score',
    'high_value_count', 'high_risk_count'
]]

print("Daily Summary:")
print(daily.to_string(index=False))

daily.to_csv(PATH_DAILY_SUMMARY, index=False)
print(f"\n✅ Saved: daily_summary.csv  ({len(daily)} rows)")


### Cell 17 — payment_method_breakdown.csv

In [ ]:
# ── Cell 17: Payment Method Breakdown ─────────────────────────────────
# One row per payment method. Powers the payment method pie/bar chart.

pm = ct.groupby('payment_method').agg(
    transaction_count  =('transaction_id', 'count'),
    total_gmv_usd      =('amount_usd',     'sum'),
    chargeback_count   =('status', lambda x: (x == 'chargeback').sum()),
    failed_count       =('status', lambda x: (x == 'failed').sum()),
    avg_risk_score     =('risk_score', 'mean'),
).reset_index()

# Captured GMV per payment method
cap_pm = ct[ct['status'] == 'captured'].groupby(
    'payment_method')['amount_usd'].sum().reset_index()
cap_pm.columns = ['payment_method', 'captured_gmv_usd']
pm = pm.merge(cap_pm, on='payment_method', how='left').fillna(0)

pm['pct_of_transactions'] = (
    pm['transaction_count'] / pm['transaction_count'].sum() * 100
).round(2)
pm['success_rate_pct'] = (
    (pm['transaction_count'] - pm['chargeback_count'] - pm['failed_count'])
    / pm['transaction_count'] * 100
).round(2)
pm['total_gmv_usd']    = pm['total_gmv_usd'].round(2)
pm['captured_gmv_usd'] = pm['captured_gmv_usd'].round(2)
pm['avg_risk_score']   = pm['avg_risk_score'].round(2)

pm = pm[[
    'payment_method', 'transaction_count', 'pct_of_transactions',
    'total_gmv_usd', 'captured_gmv_usd', 'chargeback_count',
    'failed_count', 'success_rate_pct', 'avg_risk_score'
]].sort_values('total_gmv_usd', ascending=False).reset_index(drop=True)

print("Payment Method Breakdown:")
print(pm.to_string(index=False))

pm.to_csv(PATH_PAYMENT_BREAKDOWN, index=False)
print(f"\n✅ Saved: payment_method_breakdown.csv  ({len(pm)} rows)")


### Cell 18 — region_breakdown.csv

In [ ]:
# ── Cell 18: Region Breakdown ─────────────────────────────────────────
# One row per gateway region. Powers the region bar chart.

rb = ct.groupby('gateway_region').agg(
    transaction_count =('transaction_id', 'count'),
    total_gmv_usd     =('amount_usd',     'sum'),
    chargeback_count  =('status', lambda x: (x == 'chargeback').sum()),
    failed_count      =('status', lambda x: (x == 'failed').sum()),
    avg_risk_score    =('risk_score', 'mean'),
    high_value_count  =('high_value_flag', 'sum'),
    high_risk_count   =('high_risk_flag',  'sum'),
).reset_index()

cap_rb = ct[ct['status'] == 'captured'].groupby(
    'gateway_region')['amount_usd'].sum().reset_index()
cap_rb.columns = ['gateway_region', 'captured_gmv_usd']
rb = rb.merge(cap_rb, on='gateway_region', how='left').fillna(0)

rb['success_rate_pct'] = (
    (rb['transaction_count'] - rb['chargeback_count'] - rb['failed_count'])
    / rb['transaction_count'] * 100
).round(2)
rb['chargeback_ratio_pct'] = (
    rb['chargeback_count'] / rb['transaction_count'] * 100
).round(2)
rb['total_gmv_usd']    = rb['total_gmv_usd'].round(2)
rb['captured_gmv_usd'] = rb['captured_gmv_usd'].round(2)
rb['avg_risk_score']   = rb['avg_risk_score'].round(2)

rb = rb[[
    'gateway_region', 'transaction_count', 'total_gmv_usd',
    'captured_gmv_usd', 'success_rate_pct', 'chargeback_ratio_pct',
    'avg_risk_score', 'high_value_count', 'high_risk_count',
    'chargeback_count', 'failed_count'
]].sort_values('total_gmv_usd', ascending=False).reset_index(drop=True)

print("Region Breakdown:")
print(rb.to_string(index=False))

rb.to_csv(PATH_REGION_BREAKDOWN, index=False)
print(f"\n✅ Saved: region_breakdown.csv  ({len(rb)} rows)")


### Cell 19 — merchant_performance_summary.csv

In [ ]:
# ── Cell 19: Merchant Performance Summary ─────────────────────────────
# One row per merchant. Powers the detail table in the dashboard.

mp = ct.groupby([
    'merchant_id', 'merchant_name', 'merchant_category', 'gateway_region'
]).agg(
    total_transactions =('transaction_id', 'count'),
    total_gmv_usd      =('amount_usd',     'sum'),
    chargeback_count   =('status', lambda x: (x == 'chargeback').sum()),
    failed_count       =('status', lambda x: (x == 'failed').sum()),
    avg_risk_score     =('risk_score', 'mean'),
    high_value_count   =('high_value_flag', 'sum'),
    high_risk_count    =('high_risk_flag',  'sum'),
).reset_index()

cap_mp = ct[ct['status'] == 'captured'].groupby(
    'merchant_id')['amount_usd'].sum().reset_index()
cap_mp.columns = ['merchant_id', 'captured_gmv_usd']
mp = mp.merge(cap_mp, on='merchant_id', how='left').fillna(0)

mp['chargeback_ratio_pct'] = (
    mp['chargeback_count'] / mp['total_transactions'] * 100
).round(2)
mp['success_rate_pct'] = (
    (mp['total_transactions'] - mp['chargeback_count'] - mp['failed_count'])
    / mp['total_transactions'] * 100
).round(2)
mp['total_gmv_usd']    = mp['total_gmv_usd'].round(2)
mp['captured_gmv_usd'] = mp['captured_gmv_usd'].round(2)
mp['avg_risk_score']   = mp['avg_risk_score'].round(2)

mp = mp[[
    'merchant_id', 'merchant_name', 'merchant_category', 'gateway_region',
    'total_transactions', 'total_gmv_usd', 'captured_gmv_usd',
    'chargeback_count', 'chargeback_ratio_pct', 'failed_count',
    'success_rate_pct', 'avg_risk_score', 'high_value_count', 'high_risk_count'
]].sort_values('total_gmv_usd', ascending=False).reset_index(drop=True)

print("Merchant Performance Summary:")
print(mp.to_string(index=False))

mp.to_csv(PATH_MERCHANT_PERF, index=False)
print(f"\n✅ Saved: merchant_performance_summary.csv  ({len(mp)} rows)")


---
### Cell 20 — Final Output Checklist

In [ ]:
# ── Cell 20: Final Output Checklist ──────────────────────────────────
# Verify all required output files exist and have the right row counts.

import os

expected = {
    PATH_MISSING_IN_GATEWAY : (2,  "missing_in_gateway.csv"),
    PATH_MISSING_IN_LEDGER  : (1,  "missing_in_ledger.csv"),
    PATH_AMOUNT_MISMATCHES  : (2,  "amount_mismatches.csv"),
    PATH_STATUS_MISMATCHES  : (1,  "status_mismatches.csv"),
    PATH_RECON_REPORT       : (6,  "reconciliation_report.csv"),
    PATH_API_NORMALIZED     : (6,  "api_normalized.csv"),
    PATH_DAILY_SUMMARY      : (6,  "daily_summary.csv"),
    PATH_PAYMENT_BREAKDOWN  : (4,  "payment_method_breakdown.csv"),
    PATH_REGION_BREAKDOWN   : (3,  "region_breakdown.csv"),
    PATH_MERCHANT_PERF      : (5,  "merchant_performance_summary.csv"),
    PATH_SUMMARY_METRICS    : (None,"summary_metrics.json"),
}

print(f"{'File':<42} {'Expected':>8} {'Actual':>8} {'Status':>8}")
print("-" * 72)
all_ok = True
for path, (expected_rows, label) in expected.items():
    exists = os.path.exists(path)
    if not exists:
        print(f"  {label:<40} {'—':>8} {'MISSING':>8}  ❌")
        all_ok = False
        continue
    if path.endswith('.json'):
        actual = 'JSON'
        exp_str = 'JSON'
    else:
        actual = len(pd.read_csv(path))
        exp_str = str(expected_rows)
        if actual != expected_rows:
            all_ok = False
    status = '✅' if (path.endswith('.json') or actual == expected_rows) else '⚠️'
    print(f"  {label:<40} {str(exp_str):>8} {str(actual):>8}  {status}")

print()
if all_ok:
    print("All output files verified ✅  — pipeline complete.")
else:
    print("Some files need attention ⚠️  — check output above.")
